In [27]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import csv

## Dummy Data :)

In [28]:
df_citizen = pd.read_csv(
    "../raw/citizen_reports_test.csv",
    sep=",",
    engine="python",
    quoting=csv.QUOTE_NONE
)

df_citizen.columns = df_citizen.columns.str.strip('"')

for col in df_citizen.select_dtypes(include="object").columns:
    df_citizen[col] = df_citizen[col].str.strip('"')

df_citizen.head()

,title,description,category,latitude,longitude,country,city,urgency,status,related_species
0,Plastic waste near canal,Large amount of plastic waste found near a wat...,pollution,19.286,-99.103,Mexico,Mexico City,medium,unverified,Axolotl
1,Wetland vegetation removed,Part of the vegetation near the wetland appear...,habitat_damage,19.292,-99.099,Mexico,Mexico City,high,unverified,Axolotl
2,Smoke near forest edge,Visible smoke reported near a forested zone.,fire,19.500,-99.300,Mexico,Naucalpan,high,unverified,
3,Injured bird spotted,Small injured bird found near an urban park. N...,injured_animal,19.432,-99.133,Mexico,Mexico City,medium,unverified,
4,Trash accumulation on beach,Plastic bottles and fishing line reported near...,pollution,20.629,-87.073,Mexico,Playa del Carmen,high,unverified,Hawksbill turtle


In [29]:
from pathlib import Path
path = Path("../raw/species_reference.csv")

lines = path.read_text(encoding="utf-8-sig").splitlines()

def clean_line(line):
    line = line.strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    return line.replace('""', '"')

header = clean_line(lines[0]).split(",")

rows = []

for line in lines[1:]:
    parts = clean_line(line).split(",")

    if len(parts) == len(header):
        rows.append(parts)

    elif len(parts) > len(header):
        left = parts[:8]

        right = parts[-4:]

        middle = parts[8:-4]

        conservation_note = middle[0]
        short_description = ",".join(middle[1:])

        rows.append(left + [conservation_note, short_description] + right)

    else:
        print("Bad row:", len(parts), line)

df_species = pd.DataFrame(rows, columns=header)

df_species.columns = df_species.columns.str.strip()

for col in df_species.select_dtypes(include="object").columns:
    df_species[col] = df_species[col].str.strip()

df_species.head()

,common_name,scientific_name,kingdom,class_name,order_name,family,genus,taxon_rank,conservation_note,short_description,region,source,license,attribution
0,Axolotl,Ambystoma mexicanum,Animalia,Amphibia,Urodela,Ambystomatidae,Ambystoma,Species,Critically endangered,Freshwater salamander native to the lake syste...,Mexico,GBIF,CC BY 4.0,Species taxonomy data provided by GBIF Backbon...
1,Vaquita,Phocoena sinus,Animalia,Mammalia,Artiodactyla,Phocoenidae,Phocoena,Species,Critically endangered,Small porpoise endemic to the northern Gulf of...,Mexico,GBIF,CC BY 4.0,Species taxonomy data provided by GBIF Backbon...
2,Jaguar,Panthera onca,Animalia,Mammalia,Carnivora,Felidae,Panthera,Species,Near threatened,Large feline found across Latin America and th...,Latin America,GBIF,CC BY 4.0,Species taxonomy data provided by GBIF Backbon...
3,Hawksbill turtle,Eretmochelys imbricata,Animalia,Reptilia,Testudines,Cheloniidae,Eretmochelys,Species,Critically endangered,"Marine turtle threatened by habitat loss, trad...",Tropical oceans,GBIF,CC BY 4.0,Species taxonomy data provided by GBIF Backbon...
4,Monarch butterfly,Danaus plexippus,Animalia,Insecta,Lepidoptera,Nymphalidae,Danaus,Species,Threatened by habitat loss,Iconic migratory butterfly affected by climate...,North America,GBIF,CC BY 4.0,Species taxonomy data provided by GBIF Backbon...


## Fire NASA DATA 
Source: NASA-FIRMS
Start Date :  13-01-2026
End Date : 21-06-2026
Area : Global

In [30]:
df_fire_nasa = pd.read_csv(
    "../raw/fire_nasa.csv",
    sep=",",
    engine="python",
    quoting=csv.QUOTE_NONE
)

df_fire_nasa.columns = df_fire_nasa.columns.str.strip('"')

for col in df_fire_nasa.select_dtypes(include="object").columns:
    df_fire_nasa[col] = df_fire_nasa[col].str.strip('"')

df_fire_nasa.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,7.88035,24.78273,306.03,0.45,0.39,2026-01-13,2,N21,VIIRS,n,2.0NRT,289.47,0.71,N
1,8.49771,20.88166,299.32,0.39,0.36,2026-01-13,2,N21,VIIRS,n,2.0NRT,286.98,0.24,N
2,14.46415,22.96063,311.71,0.39,0.36,2026-01-13,2,N21,VIIRS,n,2.0NRT,287.36,0.86,N
3,14.19043,23.44768,302.43,0.38,0.36,2026-01-13,2,N21,VIIRS,n,2.0NRT,287.48,0.60,N
4,11.97006,35.55463,310.75,0.64,0.72,2026-01-13,2,N21,VIIRS,n,2.0NRT,287.82,1.63,N


In [31]:
df_fire_nasa = df_fire_nasa.drop(columns=["scan", "track", "satellite", "instrument", "version"])
df_fire_nasa.head()

,latitude,longitude,brightness,acq_date,acq_time,confidence,bright_t31,frp,daynight
0,7.88035,24.78273,306.03,2026-01-13,2,n,289.47,0.71,N
1,8.49771,20.88166,299.32,2026-01-13,2,n,286.98,0.24,N
2,14.46415,22.96063,311.71,2026-01-13,2,n,287.36,0.86,N
3,14.19043,23.44768,302.43,2026-01-13,2,n,287.48,0.60,N
4,11.97006,35.55463,310.75,2026-01-13,2,n,287.82,1.63,N


In [32]:
df_fire_app = df_fire_nasa[
    [
    "latitude", 
    "longitude",
    "acq_date",
    "acq_time",
    "confidence",
    "frp",
    "brightness",
    "daynight"
    ]
].copy()
df_fire_app.head()

,latitude,longitude,acq_date,acq_time,confidence,frp,brightness,daynight
0,7.88035,24.78273,2026-01-13,2,n,0.71,306.03,N
1,8.49771,20.88166,2026-01-13,2,n,0.24,299.32,N
2,14.46415,22.96063,2026-01-13,2,n,0.86,311.71,N
3,14.19043,23.44768,2026-01-13,2,n,0.60,302.43,N
4,11.97006,35.55463,2026-01-13,2,n,1.63,310.75,N


In [35]:
df_fire_app["acq_time"] = df_fire_app["acq_time"].astype(str).str.zfill(4)

df_fire_app["detected_at_utc"] = pd.to_datetime(
    df_fire_app["acq_date"].astype(str) + " " +
    df_fire_app["acq_time"].str[:2] + ":" +
    df_fire_app["acq_time"].str[2:],
    errors="coerce"
)

def classify_fire_intensity(frp):
    if frp < 1:
        return "low"
    elif frp < 5:
        return "medium"
    else:
        return "high"

df_fire_app["fire_intensity"] = df_fire_app["frp"].apply(classify_fire_intensity)

df_fire_app = df_fire_app.drop(columns=["acq_date", "acq_time"])

df_fire_app.head()

,latitude,longitude,confidence,frp,brightness,daynight,detected_at_utc,fire_intensity
0,7.88035,24.78273,n,0.71,306.03,N,2026-01-13 00:02:00,low
1,8.49771,20.88166,n,0.24,299.32,N,2026-01-13 00:02:00,low
2,14.46415,22.96063,n,0.86,311.71,N,2026-01-13 00:02:00,low
3,14.19043,23.44768,n,0.60,302.43,N,2026-01-13 00:02:00,low
4,11.97006,35.55463,n,1.63,310.75,N,2026-01-13 00:02:00,medium


In [37]:
def_fire_high = df_fire_app[df_fire_app["fire_intensity"] == "high"]
def_fire_high.head()

,latitude,longitude,confidence,frp,brightness,daynight,detected_at_utc,fire_intensity
27,12.89155,30.08132,n,6.70,337.70,N,2026-01-13 00:02:00,high
28,12.89068,30.08611,n,15.30,308.62,N,2026-01-13 00:02:00,high
29,12.88614,30.08520,n,15.30,352.85,N,2026-01-13 00:02:00,high
30,12.88701,30.08041,n,6.70,301.26,N,2026-01-13 00:02:00,high
31,12.88159,30.08430,n,17.42,341.59,N,2026-01-13 00:02:00,high


In [38]:
processed_path = Path("../processed")
processed_path.mkdir(parents=True, exist_ok=True)

In [39]:
df_fire_app.to_csv("../processed/fire_nasa_clean.csv", index=False)

## Taxonomy Data
Source: GBIF Backbone Taxonomy

In [51]:
df_gbif_taxonomy = pd.read_csv(
    "../raw/dataset-53147.tsv",
    sep="\t",
    encoding="utf-8",
    low_memory=False
)

df_gbif_taxonomy.head()

,dwc:taxonID,dwc:parentNameUsageID,dwc:acceptedNameUsageID,dwc:taxonomicStatus,dwc:taxonRank,dwc:scientificName,dwc:scientificNameAuthorship,dwc:kingdom,dwc:phylum,dwc:class,...,dwc:superfamily,dwc:family,dwc:subfamily,dwc:tribe,dwc:subtribe,dwc:genus,dwc:subgenus,dwc:higherClassification,clb:taxGroupFromName,clb:taxGroup
0,1,NaN,NaN,accepted,kingdom,Animalia,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,animals,animals
1,4744128,1.0,NaN,provisionally accepted,genus,Ababactus,"Sharp, 1885",Animalia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Animalia,NaN,animals
2,4760454,1.0,NaN,accepted,genus,Abactrus,"Sharp, 1895",Animalia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Animalia,NaN,animals
3,11341330,4760454.0,NaN,accepted,species,Abactrus championi,"Sharp, 1895",Animalia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Abactrus,NaN,Animalia;Abactrus,NaN,animals
4,3253611,1.0,NaN,accepted,genus,Abalius,"Kraepelin, 1897",Animalia,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Animalia,NaN,animals


In [52]:
df_taxonomy = df_gbif_taxonomy.copy()

df_taxonomy.columns = (
    df_taxonomy.columns
    .str.replace("dwc:", "", regex=False)
    .str.replace("clb:", "", regex=False)
    .str.strip()
)

df_taxonomy.columns.tolist()

['taxonID',
 'parentNameUsageID',
 'acceptedNameUsageID',
 'taxonomicStatus',
 'taxonRank',
 'scientificName',
 'scientificNameAuthorship',
 'kingdom',
 'phylum',
 'class',
 'order',
 'superfamily',
 'family',
 'subfamily',
 'tribe',
 'subtribe',
 'genus',
 'subgenus',
 'higherClassification',
 'taxGroupFromName',
 'taxGroup']

In [53]:
useful_taxonomy_cols = [
    "taxonID",
    "parentNameUsageID",
    "acceptedNameUsageID",
    "taxonomicStatus",
    "taxonRank",
    "scientificName",
    "scientificNameAuthorship",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "higherClassification",
    "taxGroup"
]

existing_cols = [col for col in useful_taxonomy_cols if col in df_taxonomy.columns]

df_taxonomy_app = df_taxonomy[existing_cols].copy()

df_taxonomy_app.head()

,taxonID,parentNameUsageID,acceptedNameUsageID,taxonomicStatus,taxonRank,scientificName,scientificNameAuthorship,kingdom,phylum,class,order,family,genus,higherClassification,taxGroup
0,1,NaN,NaN,accepted,kingdom,Animalia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,animals
1,4744128,1.0,NaN,provisionally accepted,genus,Ababactus,"Sharp, 1885",Animalia,NaN,NaN,NaN,NaN,NaN,Animalia,animals
2,4760454,1.0,NaN,accepted,genus,Abactrus,"Sharp, 1895",Animalia,NaN,NaN,NaN,NaN,NaN,Animalia,animals
3,11341330,4760454.0,NaN,accepted,species,Abactrus championi,"Sharp, 1895",Animalia,NaN,NaN,NaN,NaN,Abactrus,Animalia;Abactrus,animals
4,3253611,1.0,NaN,accepted,genus,Abalius,"Kraepelin, 1897",Animalia,NaN,NaN,NaN,NaN,NaN,Animalia,animals


In [54]:
for col in df_taxonomy_app.select_dtypes(include="object").columns:
    df_taxonomy_app[col] = df_taxonomy_app[col].str.strip()

In [55]:
id_cols = ["taxonID", "parentNameUsageID", "acceptedNameUsageID"]

for col in id_cols:
    if col in df_taxonomy_app.columns:
        df_taxonomy_app[col] = pd.to_numeric(
            df_taxonomy_app[col],
            errors="coerce"
        ).astype("Int64")

In [56]:
if "taxGroup" in df_taxonomy_app.columns:
    df_taxonomy_app = df_taxonomy_app[
        df_taxonomy_app["taxGroup"].str.lower() == "animals"
    ]

In [57]:
df_taxonomy_app = df_taxonomy_app.dropna(subset=["scientificName"])

In [58]:
df_species_catalog = df_taxonomy_app[
    df_taxonomy_app["taxonRank"].str.lower().isin(["species", "subspecies"])
].copy()

df_species_catalog = df_species_catalog[
    df_species_catalog["taxonomicStatus"].str.lower().isin(
        ["accepted", "provisionally accepted"]
    )
]

df_species_catalog.head()

,taxonID,parentNameUsageID,acceptedNameUsageID,taxonomicStatus,taxonRank,scientificName,scientificNameAuthorship,kingdom,phylum,class,order,family,genus,higherClassification,taxGroup
3,11341330,4760454,<NA>,accepted,species,Abactrus championi,"Sharp, 1895",Animalia,NaN,NaN,NaN,NaN,Abactrus,Animalia;Abactrus,animals
5,6895026,3253611,<NA>,accepted,species,Abalius manilanus,"Kraepelin, 1900",Animalia,NaN,NaN,NaN,NaN,Abalius,Animalia;Abalius,animals
6,6895024,3253611,<NA>,accepted,species,Abalius rohdei,"Kraepelin, 1897",Animalia,NaN,NaN,NaN,NaN,Abalius,Animalia;Abalius,animals
7,6895023,3253611,<NA>,accepted,species,Abalius samoanus,"Kraepelin, 1897",Animalia,NaN,NaN,NaN,NaN,Abalius,Animalia;Abalius,animals
11,9089046,4810305,<NA>,accepted,species,Abessebdella palmyrae,"Richardson, 1975",Animalia,NaN,NaN,NaN,NaN,Abessebdella,Animalia;Abessebdella,animals


In [59]:
print(df_species_catalog.shape)
df_species_catalog[[
    "taxonID",
    "taxonRank",
    "scientificName",
    "scientificNameAuthorship",
    "family",
    "genus",
    "taxonomicStatus"
]].head(20)

(8611, 15)


,taxonID,taxonRank,scientificName,scientificNameAuthorship,family,genus,taxonomicStatus
3,11341330,species,Abactrus championi,"Sharp, 1895",NaN,Abactrus,accepted
5,6895026,species,Abalius manilanus,"Kraepelin, 1900",NaN,Abalius,accepted
6,6895024,species,Abalius rohdei,"Kraepelin, 1897",NaN,Abalius,accepted
7,6895023,species,Abalius samoanus,"Kraepelin, 1897",NaN,Abalius,accepted
11,9089046,species,Abessebdella palmyrae,"Richardson, 1975",NaN,Abessebdella,accepted
15,2461300,species,Ablepharis kitaibelii,"Bibron & Bory de Saint-Vincent, 1833",NaN,Ablepharis,accepted
21,12190446,species,Abyssinotus sabae,"Quéinnec & Ollivier, 2021",NaN,Abyssinotus,accepted
22,12263875,species,Abyssinotus salomon,"Quéinnec & Ollivier, 2021",NaN,Abyssinotus,accepted
25,8528860,species,Acaeniotyle gedrangta,"Empson-Morin, 1981",NaN,Acaeniotyle,accepted
26,8540941,species,Acaeniotyle parva,"Yang, 1993",NaN,Acaeniotyle,accepted


In [60]:
df_taxonomy_app.to_csv(
    "../processed/animalia_taxonomy_clean.csv",
    index=False
)

df_species_catalog.to_csv(
    "../processed/animalia_species_catalog.csv",
    index=False
)

## Fix datasets to update in Supabase

In [61]:
df_fire = pd.read_csv("../processed/fire_nasa_clean.csv")
df_species_catalog = pd.read_csv("../processed/animalia_species_catalog.csv")
df_taxonomy = pd.read_csv("../processed/animalia_taxonomy_clean.csv")

print("FIRE NASA")
print(df_fire.shape)
print(df_fire.columns.tolist())

print("\nSPECIES CATALOG")
print(df_species_catalog.shape)
print(df_species_catalog.columns.tolist())

print("\nTAXONOMY")
print(df_taxonomy.shape)
print(df_taxonomy.columns.tolist())

FIRE NASA
(5993787, 8)
['latitude', 'longitude', 'confidence', 'frp', 'brightness', 'daynight', 'detected_at_utc', 'fire_intensity']

SPECIES CATALOG
(8611, 15)
['taxonID', 'parentNameUsageID', 'acceptedNameUsageID', 'taxonomicStatus', 'taxonRank', 'scientificName', 'scientificNameAuthorship', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'higherClassification', 'taxGroup']

TAXONOMY
(19216, 15)
['taxonID', 'parentNameUsageID', 'acceptedNameUsageID', 'taxonomicStatus', 'taxonRank', 'scientificName', 'scientificNameAuthorship', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'higherClassification', 'taxGroup']


In [74]:
print(df_citizen.shape)
print(df_citizen.columns.tolist())
df_citizen.to_csv("../processed/citizen_reports.csv", index=False)
df_species.to_csv("../processed/species_test.csv", index=False)

(6, 10)
['title', 'description', 'category', 'latitude', 'longitude', 'country', 'city', 'urgency', 'status', 'related_species']


In [75]:
rename_taxonomy_cols = {
    "taxonID": "taxon_id",
    "parentNameUsageID": "parent_name_usage_id",
    "acceptedNameUsageID": "accepted_name_usage_id",
    "taxonomicStatus": "taxonomic_status",
    "taxonRank": "taxon_rank",
    "scientificName": "scientific_name",
    "scientificNameAuthorship": "scientific_name_authorship",
    "class": "class_name",
    "order": "order_name",
    "higherClassification": "higher_classification",
    "taxGroup": "tax_group"
}

df_taxonomy = df_taxonomy.rename(columns=rename_taxonomy_cols)
df_species_catalog = df_species_catalog.rename(columns=rename_taxonomy_cols)

In [79]:
id_cols = ["taxon_id", "parent_name_usage_id"]

for df in [df_taxonomy, df_species_catalog]:
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

In [80]:
print(df_species_catalog.columns.tolist())

print(df_taxonomy.columns.tolist())

['taxon_id', 'parent_name_usage_id', 'taxonomic_status', 'taxon_rank', 'scientific_name', 'scientific_name_authorship', 'kingdom', 'phylum', 'class_name', 'order_name', 'family', 'genus', 'higher_classification', 'tax_group']
['taxon_id', 'parent_name_usage_id', 'taxonomic_status', 'taxon_rank', 'scientific_name', 'scientific_name_authorship', 'kingdom', 'phylum', 'class_name', 'order_name', 'family', 'genus', 'higher_classification', 'tax_group']


In [81]:
df_taxonomy.to_csv(processed_path / "animalia_taxonomy_clean.csv", index=False)
df_species_catalog.to_csv(processed_path / "animalia_species_catalog.csv", index=False)

NASA DATA HAS over 6 million datasets

In [67]:
df_fire["fire_intensity"].value_counts()

fire_intensity
high      3274054
medium    2391351
low        328382
Name: count, dtype: int64

In [70]:
sample_size = 300

df_fire_sample = (
    df_fire
    .groupby("fire_intensity", group_keys=False)
    .apply(lambda x: x.sample(n=min(sample_size, len(x)), random_state=42))
    .reset_index(drop=True)
)

df_fire_sample.head()

C:\Users\Esme\AppData\Local\Temp\ipykernel_22452\2072191578.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(sample_size, len(x)), random_state=42))


,latitude,longitude,confidence,frp,brightness,daynight,detected_at_utc,fire_intensity
0,6.59409,-69.05666,n,5.47,341.31,D,2026-03-04 17:15:00,high
1,49.68079,18.65973,n,7.37,330.31,N,2026-01-18 01:38:00,high
2,11.36359,3.95322,n,7.47,337.39,D,2026-02-17 13:33:00,high
3,6.89423,18.65497,n,6.72,338.55,D,2026-02-21 12:16:00,high
4,16.52172,106.43051,l,16.63,338.21,D,2026-04-08 06:12:00,high


In [71]:
df_fire_sample["fire_intensity"].value_counts()

fire_intensity
high      300
low       300
medium    300
Name: count, dtype: int64

In [72]:
df_fire_sample = df_fire_sample.sort_values(
    by="detected_at_utc",
    ascending=False
).reset_index(drop=True)

df_fire_sample.head()

,latitude,longitude,confidence,frp,brightness,daynight,detected_at_utc,fire_intensity
0,-11.97960,17.64441,n,6.04,354.12,D,2026-06-21 13:01:00,high
1,-5.69498,27.11269,n,7.96,345.87,D,2026-06-21 11:21:00,high
2,-7.26878,34.46854,l,4.80,333.70,D,2026-06-21 11:21:00,medium
3,59.73135,90.46096,n,5.61,336.84,D,2026-06-21 06:35:00,high
4,59.50425,90.30357,n,4.00,330.81,D,2026-06-21 06:35:00,medium


In [ ]:
df_fire_sample.to_csv(
    "../processed/fire_nasa_sample_900.csv",
    index=False
)